[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-4/map-reduce.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239947-lesson-3-map-reduce)

# Map-reduce

## 回顾

我们正在构建一个多智能体研究助理，它将本课程所有模块的内容整合在一起。

为了构建这个多智能体助理，我们已经介绍了一些 LangGraph 的可控性主题。

我们刚刚讲解了并行化和子图。

## 目标

现在，我们将介绍 [map reduce](https://langchain-ai.github.io/langgraph/how-tos/map-reduce/)。

In [ ]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我们将使用 [LangSmith](https://docs.smith.langchain.com/) 进行[跟踪](https://docs.smith.langchain.com/concepts/tracing)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

## 问题

Map-reduce 操作对于高效的任务分解和并行处理至关重要。

它有两个阶段：

(1) `Map` - 将任务分解为更小的子任务，并行处理每个子任务。

(2) `Reduce` - 聚合所有已完成并行化子任务的结果。

让我们设计一个系统来做两件事：

(1) `Map` - 创建一组关于某个主题的笑话。

(2) `Reduce` - 从列表中选择最好的笑话。

我们将使用 LLM 来完成笑话生成和选择。

In [ ]:
from langchain_openai import ChatOpenAI

# 我们将使用的提示词
subjects_prompt = """Generate a list of 3 sub-topics that are all related to this overall topic: {topic}."""
joke_prompt = """Generate a joke about {subject}"""
best_joke_prompt = """Below are a bunch of jokes about {topic}. Select the best one! Return the ID of the best one, starting 0 as the ID for the first joke. Jokes: \n\n  {jokes}"""

# LLM
model = ChatOpenAI(model="gpt-4o", temperature=0) 

## 状态

### 并行化笑话生成

首先，让我们定义图的入口点，它将：

* 接受用户输入的主题
* 从中产生笑话主题列表
* 将每个笑话主题发送到我们上面的笑话生成节点

我们的状态有一个 `jokes` 键，它将累积来自并行化笑话生成的笑话

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel

class Subjects(BaseModel):
    subjects: list[str]

class BestJoke(BaseModel):
    id: int
    
class OverallState(TypedDict):
    topic: str
    subjects: list
    jokes: Annotated[list, operator.add]
    best_selected_joke: str

生成笑话的主题。

In [ ]:
def generate_topics(state: OverallState):
    prompt = subjects_prompt.format(topic=state["topic"])
    response = model.with_structured_output(Subjects).invoke(prompt)
    return {"subjects": response.subjects}

这里是关键所在：我们使用 [Send](https://langchain-ai.github.io/langgraph/concepts/low_level/#send) 为每个主题创建一个笑话。

这非常有用！它可以自动并行化任意数量主题的笑话生成。

* `generate_joke`: 图中节点的名称
* `{"subject": s}`: 要发送的状态

`Send` 允许您将任何您想要的状态传递给 `generate_joke`！它不必与 `OverallState` 对齐。

在这种情况下，`generate_joke` 使用自己的内部状态，我们可以通过 `Send` 来填充它。

In [ ]:
from langgraph.types import Send
def continue_to_jokes(state: OverallState):
    return [Send("generate_joke", {"subject": s}) for s in state["subjects"]]

### 笑话生成 (map)

现在，我们只需定义一个创建笑话的节点 `generate_joke`！

我们将它们写回到 `OverallState` 中的 `jokes`！

这个键有一个 reducer，将组合列表。

In [ ]:
class JokeState(TypedDict):
    subject: str

class Joke(BaseModel):
    joke: str

def generate_joke(state: JokeState):
    prompt = joke_prompt.format(subject=state["subject"])
    response = model.with_structured_output(Joke).invoke(prompt)
    return {"jokes": [response.joke]}

### 最佳笑话选择 (reduce)

现在，我们添加选择最佳笑话的逻辑。

In [ ]:
def best_joke(state: OverallState):
    jokes = "\n\n".join(state["jokes"])
    prompt = best_joke_prompt.format(topic=state["topic"], jokes=jokes)
    response = model.with_structured_output(BestJoke).invoke(prompt)
    return {"best_selected_joke": state["jokes"][response.id]}

## 编译

In [ ]:
from IPython.display import Image
from langgraph.graph import END, StateGraph, START

# 构建图：在这里我们将所有内容组合在一起构建我们的图
graph = StateGraph(OverallState)
graph.add_node("generate_topics", generate_topics)
graph.add_node("generate_joke", generate_joke)
graph.add_node("best_joke", best_joke)
graph.add_edge(START, "generate_topics")
graph.add_conditional_edges("generate_topics", continue_to_jokes, ["generate_joke"])
graph.add_edge("generate_joke", "best_joke")
graph.add_edge("best_joke", END)

# 编译图
app = graph.compile()
Image(app.get_graph().draw_mermaid_png())

In [ ]:
# 调用图：在这里我们调用它来生成笑话列表
for s in app.stream({"topic": "animals"}):
    print(s)

## Studio

**⚠️ 免责声明**

自这些视频录制以来，我们已经更新了 Studio，使其可以在本地运行并在浏览器中打开。这现在是运行 Studio 的首选方式（而不是像视频中显示的使用桌面应用程序）。请参阅有关本地开发服务器的文档[这里](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server)和[这里](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server)。要启动本地开发服务器，请在此模块的 `/studio` 目录中在终端中运行以下命令：

```
langgraph dev
```

您应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到 Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

让我们在 Studio UI 中加载上面的图，它使用在 `module-4/studio/langgraph.json` 中设置的 `module-4/studio/map_reduce.py`。